In [ ]:
# Imports 

import torch                     # main PyTorch library for tensors and deep learning
import torch.nn as nn            # neural network modules (layers, models, etc.)
from torch.nn import functional as F   # functional API (activations, loss functions, etc.)

In [ ]:
# Hyperparameters

batch_size = 64                # number of sequences per batch
block_size = 256               # max context length (tokens per sequence)
max_iters = 5000               # total training iterations
eval_interval = 1000            # evaluate loss every 500 steps
learning_rate = 3e-4           # step size for optimizer (0.0003)
device = "cuda" if torch.cuda.is_available() else "cpu"  # use GPU if available
eval_iters = 50               # iterations to average loss during eval
n_embd = 384                   # embedding dimension size
n_head = 6                     # number of attention heads
n_layer = 6                    # number of transformer layers
dropout = 0.2                  # dropout rate for regularization

"""
Frequent Evaluation

You have eval_interval = 500 and eval_iters = 200. This means every 500 steps, the model stops training to run 200 forward passes on validation data.

The Fix: If you just want to see if it's learning, reduce eval_iters to 50 or increase eval_interval to 1000.
"""

In [ ]:
torch.manual_seed(1337)   # set random seed for reproducibility

# Read the file 
with open('input.txt', 'r', encoding='utf-8') as f:  # open text file in read mode
    text = f.read()       # read entire file contents into a string

In [ ]:
# unique characters
chars = sorted(list(set(text)))          # get all unique characters from text
vocab_size = len(chars)                  # size of vocabulary
print(chars)                             # print the list of characters

# mapping
stoi = {ch: i for i, ch in enumerate(chars)}   # char → integer (string to index)
itos = {i: ch for i, ch in enumerate(chars)}   # integer → char (index to string)

encode = lambda s: [stoi[c] for c in s]        # encode string into list of ints
decode = lambda l: ''.join([itos[i] for i in l]) # decode list of ints back to string

In [ ]:
# Train and validation split

data = torch.tensor(encode(text), dtype=torch.long)  # convert entire text into tensor of token IDs
n = int(0.9 * len(data))                            # use 90% of data for training
train_data = data[:n]                               # first 90% → training set
val_data = data[n:]                                 # last 10% → validation set

In [ ]:
# load the data 
def get_batch(split):
    # choose train or validation data
    data = train_data if split == 'train' else val_data 
    
    # random starting indices for batch
    ix = torch.randint(len(data) - block_size, (batch_size,))
    
    # create input (x) and target (y) sequences
    x = torch.stack([data[i:i+block_size] for i in ix])       # (B, T)
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])   # (B, T)
    
    # move to device (CPU/GPU)
    x, y = x.to(device), y.to(device)
    return x, y

In [ ]:
# Attention head 
class Head(nn.Module):   # capitalized class name (convention)
    # one head of self-attention
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)    # project input to key
        self.query = nn.Linear(n_embd, head_size, bias=False)  # project input to query
        self.value = nn.Linear(n_embd, head_size, bias=False)  # project input to value
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))  # causal mask
        self.dropout = nn.Dropout(dropout)                     # dropout for regularization
        
    def forward(self, x):
        # input: (B, T, C) → batch, time steps, channels
        B, T, C = x.shape
        
        # project input into key, query, value
        k = self.key(x)    # (B, T, HS)
        q = self.query(x)  # (B, T, HS)
        v = self.value(x)  # (B, T, HS)
        
        # compute attention scores (scaled dot-product)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)   # (B, T, T)
        
        # apply causal mask (prevent attending to future tokens)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))  # (B, T, T)
        
        # normalize scores into probabilities
        wei = F.softmax(wei, dim=-1)   # (B, T, T)
        wei = self.dropout(wei)        # apply dropout
        
        # weighted sum of values
        out = wei @ v   # (B, T, T) @ (B, T, HS) → (B, T, HS)
        return out

In [ ]:
# MultiheadAttention
class MultiHeadAttention(nn.Module):
    # multiple heads of attention in parallel 
    def __init__(self, n_heads, head_size):
        super().__init__()
        # create a list of attention heads
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_heads)])
        # final linear projection to combine heads back into embedding dimension
        self.proj = nn.Linear(head_size * n_heads, n_embd)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        # run input through each head and concatenate results along channel dimension
        out = torch.cat([h(x) for h in self.heads], dim=-1)   # (B, T, head_size * n_heads)
        # project back to embedding dimension and apply dropout
        out = self.dropout(self.proj(out))                    # (B, T, n_embd)
        return out

In [ ]:
# FeedForward 
class FeedForward(nn.Module):
    # a simple linear layer followed by non-linearity
    def __init__(self, n_embd):
        super().__init__()   # ✅ call the parent constructor properly
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),   # expand embedding dimension
            nn.ReLU(),                       # apply non-linearity
            nn.Linear(4 * n_embd, n_embd),   # project back to embedding dimension
            nn.Dropout(dropout)              # regularization
        )
        
    def forward(self, x):
        return self.net(x)   # pass input through feedforward network

In [ ]:
# Block
class Block(nn.Module):
    # transformer block: communication (attention) followed by computation (feedforward)
    def __init__(self, n_embd, n_head):   # n_embd = embedding dim, n_head = number of heads
        super().__init__()
        head_size = n_embd // n_head
        # multi-head attention should take n_heads and head_size
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        # layer normalization before each sub-layer
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        
    def forward(self, x):
        # residual connection around self-attention
        x = x + self.sa(self.ln1(x))
        # residual connection around feedforward
        x = x + self.ffwd(self.ln2(x))
        return x

In [ ]:
# GPTLanguageModel 
class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        # token embedding: maps vocab indices → embedding vectors
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        # positional embedding: encodes position in sequence
        self.positional_embedding_table = nn.Embedding(block_size, n_embd)
        # stack of transformer blocks
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)          # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)  # output layer → vocab logits
        
        # initialize weights
        self.apply(self.init_weights)
        
    def init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)   # ✅ use normal_ (in-place)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)                      # ✅ bias should be zeros
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)   # ✅ correct init for embeddings
                
    def forward(self, idx, targets=None):
        B, T = idx.shape   # batch size, sequence length
        
        # embeddings
        tok_emb = self.token_embedding_table(idx)                      # (B, T, C)
        pos_emb = self.positional_embedding_table(torch.arange(T, device=device))  # (T, C)
        x = tok_emb + pos_emb                                          # (B, T, C)
        
        # transformer blocks
        x = self.blocks(x)                                             # (B, T, C)
        x = self.ln_f(x)                                               # (B, T, C)
        
        # final logits
        logits = self.lm_head(x)                                       # (B, T, vocab_size)
        
        # compute loss if targets provided
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)                               # flatten
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)                    # cross-entropy loss
            
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens (context window)
            idx_cond = idx[:, -block_size:]
            
            # get predictions from the model
            logits, _ = self(idx_cond)              # forward pass
            
            # focus only on the last time step
            logits = logits[:, -1, :]               # (B, C)
            
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1)       # (B, C)
            
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        
        return idx

In [ ]:
# create an instance of the GPT language model
model = GPTLanguageModel()

# move the model to the chosen device (CPU or GPU)
m = model.to(device)
m = torch.compile(m)
# print the total number of parameters in millions
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

In [ ]:
# estimate loss 
@torch.no_grad()                     # disable gradient tracking (faster, less memory)
def estimate_loss():
    out = {}                         # dictionary to store average losses
    model.eval()                     # set model to evaluation mode (no dropout, etc.)
    for split in ['train', 'val']:   # loop over both train and validation sets
        losses = torch.zeros(eval_iters)   # tensor to collect losses
        for k in range(eval_iters):        # run eval_iters times
            X, Y = get_batch(split)        # get a batch of data
            logits, loss = model(X, Y)     # forward pass, compute loss
            losses[k] = loss.item()        # store loss value
        out[split] = losses.mean()         # average loss for this split
    model.train()                          # reset model back to training mode
    return out                             # return dict with train/val lossess

In [ ]:
# Optimizer 

# create a PyTorch optimizer (AdamW with given learning rate)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every eval_interval steps (or at the very end), check train/val loss
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of training data
    xb, yb = get_batch('train')

    # forward pass: compute logits and loss
    logits, loss = model(xb, yb)

    # reset gradients before backprop
    optimizer.zero_grad(set_to_none=True)

    # backward pass: compute gradients
    loss.backward()

    # update model parameters
    optimizer.step()

In [ ]:
# generate from the trained model
context = torch.zeros((1, 1), dtype=torch.long, device=device)   # start with a single token (0)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))  # generate 500 tokens and decode

# optionally save a longer generation to a file
# open('more.txt', 'w').write(decode(m.generate(context, max_new_tokens=10000)[0].tolist()))